# Shuffling and the Random Index case

In this session, we will study the effect of shuffling. As describe into detail in the Section 6.1 of the book, shuffling consists in randomizing the order of the operations that are applied independently during an execution. 
More into the details, we will focus on the SubByte operations of the first round of the AES (running on a STMF415 at 84MHz). 

As a first educational case study, we will rely on the relaxed Random Start Index (RSI), which consists in processing the SubByte sequentially, in order, but starting at a random index taken randomly in the range [0;15]. The following cells allows you to observe the impact on the SNR when the shuffling countermeasure is turned off or on. What do you observe? Is it expected? What do you think the expected impact on the attack's performance will be? 


### SNR of the Sbox outputs without shuffling

In [1]:
# Import the useful module for the session
from utils_scale import utils_files, utils_aes, utils_plot, utils_IT, utils_ta, utils_flow
import numpy as np
from scalib.metrics import SNR

In [2]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG["sw-aes-RIoff_training"], seed_shuffle=0, remove_first=True)

In [3]:
# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()


In [4]:
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10], use_log_scale=True)

### SNR of the Sbox outputs with shuffling (RSI)

In [5]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG["sw-aes-RIon_training"], seed_shuffle=0, remove_first=True, load_rnd=True)

In [6]:
# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()

print(f"The dataset contains {ds['traces'].shape[0]} traces")

In [7]:
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10], use_log_scale=True)

## Impact on (naive) attack perfomance
As we did in the first session, we next evaluate the attacks performances using SNR as POIs selection and a Multivariate Gaussian Model. What is the best attack you can achieve in both cases? 

While you may wonder why we redo the same attacks as in the first session, you may have noticed that the traces are slightly different in the provided dataset: they were actually collected using a higher sampling rate (i.e., 125MHz instead of 62.5MHz) for the purpose of this exercice. Besides allowing a fair comparison when evaluating the effect of the countermeasure, it offers us an intersting point of comparison compared to the model of the first session: how does the information extracted by the model compare to yesterday? How would you explain that?

### With RSI shuffling turned off

In [8]:
q_t = 5000 # Training complexity
q_as = range(40) # Attack complexities for which to perfom the attacks
explo_npois = [1] # Amount of POIs kept during profiling
explo_ndims = [1] # Amount of dimension in the linear-subspace. 


utils_flow.full_TA_flow_pSB(
    q_t, 
    q_as, 
    explo_npois, 
    explo_ndims, 
    utils_files.DS_CFG['sw-aes-RIoff_training'],
    utils_files.DS_CFG['sw-aes-RIoff_atcks'],
    show_heatmap = True
)

### With RSI shuffling turned on

In [9]:
q_t = None # Training complexity
q_as = range(40) # Attack complexities for which to perfom the attacks
explo_npois = [1] # Amount of POIs kept during profiling
explo_ndims = [1] # Amount of dimension in the linear-subspace. 


utils_flow.full_TA_flow_pSB(
    q_t, 
    q_as, 
    explo_npois, 
    explo_ndims, 
    utils_files.DS_CFG['sw-aes-RIon_training'],
    utils_files.DS_CFG['sw-aes-RIon_atcks'],
    show_heatmap = True
)

## Permutation leakage and advanced attack

As every computation, those related to the operations handling the permutation leak too! It turns out we can exploit this leakage to improve our model, which will hopefully lead to better attack performances. This advanced modelling targets two components:

1. the permutation used (or in our case the RSI value),
1. the intermediate values manipulated at each step of the permuted execution (i.e., the 16 sequential values manipulated, regardless of the Sbox index to which it refers)

As described in details in Section 6.1.1 of the book (Eq. 42, 43, 44 and 45), accurate models of their leakage can be combined in order to build a more advanced model directly targetting the Sboxes output (even in the presence of shuffling). The following cells study more into the details what can be achieved. 

In [10]:
# Import the useful module for the exercise
from utils_scale import utils_files, utils_aes, utils_plot, utils_IT, utils_ta, utils_flow
import numpy as np
from scalib.metrics import SNR

### SNR about the permutation

As you're probably starting to suspect by now, the first preliminary step is to check the SNR related to the random index used. What do you observe? How do you explain it?

In [11]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG["sw-aes-RIon_training"], seed_shuffle=0, remove_first=True, load_rnd=True)


In [12]:
# Fetch the random index used for the first round.
# More into the details, the index is encoded on a random bytes, from which only the 4 first bits
# are used.
ris = (ds["rand"][:,0] & 0xf)[:,np.newaxis]

# Compute the SNR associated to each classes with SCAlib.
# Here whe only use 16 classes (rather than the 256 you used in the first session).
snrobj = SNR(nc=16)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    ris.astype(np.uint16)
)
snrs = snrobj.get_snr()

In [13]:
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10])

### Information about the permutation

Next, we dig a bit more and try to assess the amount of information that can be extracted from the trace about the RSI used. Try to fit a model achieved the best amount of extracted information. From your preliminary results, how would you qualify the efficiency of the studied RSI shuffling implementation? 

In [14]:
q_t = None
explo_npois = [1]
explo_ndims = [1]


# Load the dataset 
ds = utils_files.load_dataset(
    utils_files.DS_CFG['sw-aes-RIon_training'],
    seed_shuffle=0, 
    remove_first=True, 
    load_rnd=True, 
    cropping=None
)

# Compute the indexes used for the first round
ris = (ds["rand"][:,0] & 0xf)[:,np.newaxis]

# Identify the best params and PI for the perm
explos_params = utils_IT.explore_params_LDA(
    ds['traces'], 
    ris, 
    2048, 
    explo_npois, 
    explo_ndims, 
    q_t=q_t, 
    chunk_size=10000, 
    nclasses=16
)
utils_plot.make_heatmap(explos_params)
param_set_permutation = utils_IT.identify_best_params(explos_params)


### Information about shuffled intermediate

Next, we pursue the analysis by targetting the permuted intermediate variable. That is, for every Sbox outputs, we will apply the known permutation and used these permuted values as our profilling labels.

In [15]:
# Output of the Sboxes, in AES natural order. 
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Random indexes used
ris = (ds["rand"][:,0] & 0xf)

# Apply the shuffling used to the labels
for i, ri in enumerate(ris):
    tmp = labels[i, :ri].copy()
    labels[i,:16-ri] = labels[i, ri:]
    labels[i,16-ri:] = tmp

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10])



In [16]:
q_t = None
explo_npois = [1]
explo_ndims = [1]


# Identify the best params and PI for the permutation used
explos_params = utils_IT.explore_params_LDA(
    ds['traces'], 
    labels, 
    2048, 
    explo_npois, 
    explo_ndims, 
    q_t=q_t, 
    chunk_size=10000, 
    nclasses=256
)
utils_plot.make_heatmap(explos_params)
param_set_intermediates = utils_IT.identify_best_params(explos_params)


### How does the two combine? 

Next, we combine the models targetting the permutation and the shuffled intermediates in order to recover the information about the Sbox output in the presence of RSI shuffling. What is the amount of information you can recover about the different bytes? How does it compare to the case when the RSI is turned off? 



In [17]:
qt_s = [4096] # Training complexities to test
ntraces_testing = 5000 # Amount of indep. test traces used to estimate the PI with the trained model. 



# Output of the Sboxes, in AES natural order. 
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]
in_order_sbox_labels= labels.copy()

# Random indexes used
ris = (ds["rand"][:,0] & 0xf)

# Apply the shuffling used to the labels
for i, ri in enumerate(ris):
    tmp = labels[i, :ri].copy()
    labels[i,:16-ri] = labels[i, ri:]
    labels[i,16-ri:] = tmp

pis_curves = utils_IT.compute_PI_curves_RSI(
    ds['traces'][ntraces_testing:,:],
    ris[ntraces_testing:],
    labels[ntraces_testing:,:],
    ds['traces'][:ntraces_testing,:],
    in_order_sbox_labels[:ntraces_testing,:],
    qt_s,
    param_set_permutation,
    param_set_intermediates,
    chunk_size=10000
)

tis_curves = utils_IT.compute_TI_curves_RSI(
    ds['traces'][ntraces_testing:,:],
    ris[ntraces_testing:],
    labels[ntraces_testing:,:],
    in_order_sbox_labels[ntraces_testing:,:],
    qt_s,
    param_set_permutation,
    param_set_intermediates,
    chunk_size=10000
)

In [18]:
# Choose the byte for which you want to display the results
byte_indexes = range(16) # Put any list of index here if you want

# Display the IT curves for the bytes indexes that you want
utils_plot.display_IT_results(
    byte_indexes,
    [
        ("LDA", pis_curves),
        ("LDA", tis_curves)
    ],
    scale=0.6,
    disable_legend=True
)

### Key rank estimation exploiting combined leakage

As a cherry on the top, the following cells compute the ranks estimation when perofmring template attack with models that combine both leakages. What is the best attack complexity you can achieve? How does it compare to the unprotected case? How would you qualify RSI shuffling?

In [19]:
q_t = None
q_as = range(10)

atck_res = utils_ta.explore_TA_multivariate_best_RI(
    utils_files.DS_CFG['sw-aes-RIon_training'], 
    utils_files.DS_CFG['sw-aes-RIon_atcks'],
    q_t, 
    q_as, 
    param_set_intermediates, 
    param_set_permutation
)

In [20]:
key_bytes_ranks = range(16)
utils_plot.display_ranks_full_key([atck_res], key_bytes_ranks)

## Going further: a true permutation case

For those interested, the following cells compute the SNRs and the information about the permutation index used for each shuffled operation when a true permutation is applied (instead of a single random start index). How do you think the security compare to the RSI case? Would you conclude this solution as secure enough[¹]?

[¹]: *hint: A more practical case study can be found in Section 4 of [Security Analysis of Deterministic Re-Keying
with Masking & Shuffling: Application to ISAP](https://perso.uclouvain.be/fstandae/PUBLIS/267.pdf).*

### SNR of the Sbox outputs with shuffling

In [21]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG["sw-aes-SHon_training"], seed_shuffle=0, remove_first=True, load_rnd=True)

In [22]:
# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()

# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10], use_log_scale=True)

### SNR about the shuffled indexes

In [23]:
# Fetch the random index used for the first round.
# More into the details, the index is encoded on a random bytes, from which only the 4 first bits
# are used.
ris = (ds["rand"] & 0xf)

# Compute the SNR associated to each classes with SCAlib.
# Here whe only use 16 classes (rather than the 256 you used in the first session).
snrobj = SNR(nc=16)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    ris.astype(np.uint16)
)
snrs = snrobj.get_snr()

# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10])

### Information about the shuffled indexes

In [24]:
q_t = None
explo_npois = [32, 128, 256, 512, 1500]
explo_ndims = [1, 2, 4, 8, 16, 32]


# Load the dataset 
ds = utils_files.load_dataset(
    utils_files.DS_CFG["sw-aes-SHon_training"],
    seed_shuffle=0, 
    remove_first=True, 
    load_rnd=True, 
    cropping=None
)

# Compute the indexes used for the first round
ris = (ds["rand"] & 0xf)

# Identify the best params and PI for the perm
explos_params = utils_IT.explore_params_LDA(
    ds['traces'], 
    ris, 
    2048, 
    explo_npois, 
    explo_ndims, 
    q_t=q_t, 
    chunk_size=10000, 
    nclasses=16
)
utils_plot.make_heatmap(explos_params)
param_set_true_perm = utils_IT.identify_best_params(explos_params)
